In [1]:
!pip install -U unsloth transformers==4.56.2 trl==0.22.2

In [ ]:
%%writefile train.py
# train.py
from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch
import os


def is_main_process():
    return int(os.environ.get("RANK", "0")) == 0


def main():
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("hf")

    model, tokenizer = FastModel.from_pretrained(
        model_name="CohereLabs/tiny-aya-base",
        max_seq_length=1024,
        load_in_4bit=True,
        full_finetuning=False,
        token=hf_token,
        # do NOT set device_map="auto" for DDP
    )

    model = FastModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        random_state=3407,
        gradient_checkpointing="unsloth",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    train_dataset = load_dataset(
        "legesher/language-decoded-data",
        "condition-1-en",
        split="train",
    )
    eval_dataset = load_dataset(
        "legesher/language-decoded-data",
        "condition-1-en",
        split="validation",
    )

    def formatting_prompts_func(examples):
        return {"code": examples["code"]}

    train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
    eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

    if is_main_process():
        gpu_stats = torch.cuda.get_device_properties(0)
        start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
        max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
        print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
        print(f"{start_gpu_memory} GB of memory reserved.")

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=SFTConfig(
            dataset_text_field="code",
            max_grad_norm=1.0,

            # With 2 GPUs, global batch = per_device_train_batch_size * 2 * grad_accumulation_steps
   
            per_device_train_batch_size=4,
            gradient_accumulation_steps=2,

            warmup_ratio=0.05,
            #max_steps=1,
            num_train_epochs = 1, # Set this for 1 full training run.

            learning_rate=5e-5,
            packing=True,
            logging_steps=1,
            optim="paged_adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=42,
            report_to="none",
            fp16=True,
            bf16=False,
            eval_strategy="steps",
            eval_steps=500,
            run_name="condition-1-en",

            # usually good with DDP + LoRA
            ddp_find_unused_parameters=False,
        ),
    )

    trainer_stats = trainer.train()

    if is_main_process():
        used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
        print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
        print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
        print(f"Peak reserved memory = {used_memory} GB.")

        model.save_pretrained("output/condition-1-en")
        tokenizer.save_pretrained("output/condition-1-en")
        model.push_to_hub("legesher/language-decoded-lora", token=hf_token)


if __name__ == "__main__":
    main()

Overwriting train.py


In [ ]:
!torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py

W0321 19:13:38.811000 2162 torch/distributed/run.py:852] 
W0321 19:13:38.811000 2162 torch/distributed/run.py:852] *****************************************
W0321 19:13:38.811000 2162 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0321 19:13:38.811000 2162 torch/distributed/run.py:852] *****************************************
[W321 19:13:39.994742258 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-21 19:13:43.489767: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00